<a href="https://colab.research.google.com/github/cked007-glitch/IBVAP-Intelligent-Border-Surveillance/blob/main/ANPR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.0 MB/s eta 0:00:00


In [5]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 20.2 MB/s eta 0:00:00


In [ ]:
import glob
import cv2
import easyocr
from ultralytics import YOLO

def process_anpr_stream(video_source):
    # Initialize YOLO for vehicle detection (classes: 2=car, 3=motorcycle, 5=bus, 7=truck)
    model = YOLO("yolov8n.pt")

    # Initialize EasyOCR reader for English plates
    print("Initializing EasyOCR engine...")
    reader = easyocr.Reader(['en'], gpu=True)

    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print(f"[ERROR] Could not open video source: '{video_source}'")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS) or 30)

    out = cv2.VideoWriter('anpr_output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    print(f"--- ANPR STREAM PROCESSING INITIATED: {video_source} ---")
    frame_count = 0
    scanned_plates = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        # Track vehicles: 2 (car), 3 (motorcycle), 5 (bus), 7 (truck)
        results = model.track(frame, classes=[2, 3, 5, 7], conf=0.5, persist=True, verbose=False, vid_stride=3)

        if results[0].boxes.id is not None:
            for box, track_id in zip(results[0].boxes.xyxy, results[0].boxes.id):
                x1, y1, x2, y2 = [int(v) for v in box.tolist()]
                box_height = y2 - y1
                t_id = int(track_id)

                plate_text = "Scanning Plate..."
                box_color = (0, 255, 255) # Yellow

                # Trigger ANPR check only when the vehicle is close enough
                if box_height > 100:
                    # Crop the lower 40% of the vehicle bounding box where plates usually reside
                    plate_crop_y1 = int(y1 + (box_height * 0.6))
                    plate_crop = frame[plate_crop_y1:y2, x1:x2]

                    if plate_crop.size > 0:
                        # Run EasyOCR on the crop
                        ocr_results = reader.readtext(plate_crop)

                        for bbox, text, prob in ocr_results:
                            # Filter for high-confidence alphanumeric text matching plate patterns
                            if prob > 0.35 and len(text.strip()) >= 4:
                                cleaned_text = text.upper().strip()
                                plate_text = f"PLATE: {cleaned_text}"
                                box_color = (0, 255, 0) # Green

                                if cleaned_text not in scanned_plates:
                                    scanned_plates.add(cleaned_text)
                                    print(f"[ANPR ALERT] Frame {frame_count} | Vehicle ID {t_id} | Plate Read: {cleaned_text} (Conf: {prob:.2f})")

                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(frame, f"ID {t_id} | {plate_text}", (x1, max(20, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"\n--- ANPR SCAN COMPLETE: Processed {frame_count} frames | Unique plates logged: {len(scanned_plates)} ---")
    print("Saved output as 'anpr_output.mp4'.\n")

if __name__ == "__main__":
    video_files = glob.glob("*.mp4")
    if video_files:
        print(f"Auto-selected video: {video_files[0]}")
        process_anpr_stream(video_files[0])
    else:
        print("[ERROR] No .mp4 files found in the workspace directory.")

Auto-selected video: Ultra_sharp_static_CCTV_traff.mp4
Initializing EasyOCR engine...
--- ANPR STREAM PROCESSING INITIATED: Ultra_sharp_static_CCTV_traff.mp4 ---
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 394ms
Prepared 1 package in 52ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[ANPR ALERT] Frame 23 | Vehicle ID 1 | Plate Read: JK K (Conf: 0.47)
[ANPR ALERT] Frame 23 | Vehicle ID 1 | Plate Read: 0157 (Conf: 0.72)
[ANPR ALERT] Frame 26 | Vehicle ID 1 | Plate Read: 01571 (Conf: 0.40)
